In [1]:
source("/rd1/user/lit/project/sORFs/sORFs.utils.R")
source("~/bin/lit_utils.R")
lib_text()
lib_plot()
setwd("/rd1/user/lit/project/sORFs/analysis/20250523_human_ms_run")
output_path <- "stat_output/S4"
create_path(output_path)

read.table("/rd1/user/lit/project/sORFs/analysis/20250404_tmp_from_nanjing/uniprot.human.sep.id.txt") -> uniprot.human.sep.id
# 主要为了获取列名
read_psm("./output/db_search_20250326/Trypsin/CAD20241224licq_BSEP_DDA_60min_21pcw_1_PAGE_T_T_Slot1_46_1_4747_d/psm.tsv") -> psm
# 输入total_psm路径
system("parallel -j 30 --pipepart --block 100M -a /rd1/user/lit/project/sORFs/analysis/20250523_human_ms_run/stat_output/S2/total_psm.txt grep -f /rd1/user/lit/project/sORFs/analysis/20250404_tmp_from_nanjing/uniprot.human.sep.id.txt > stat_output/S4/psm_contain_cano_sep.txt")
fread_c("stat_output/S4/psm_contain_cano_sep.txt") -> test_psm_contain_cano_sep
test_psm_contain_cano_sep -> df

Warning message:
“程辑包‘stringr’是用R版本4.3.2 来建造的”


In [3]:
test_psm_contain_cano_sep -> df
colnames(df) <- colnames(psm)

In [4]:
# 分组并且展开
df$Group_id <- 1:nrow(df)
df %>% separate_rows(`Mapped.Proteins`, convert = TRUE,sep=", ") -> tmp
tmp %>% mutate(Protein=`Mapped.Proteins`,
               `Protein.ID`=`Mapped.Proteins`,
               Gene=`Mapped.Proteins`,
               `Entry.Name`=`Mapped.Proteins`,
               `Protein.Description`=`Mapped.Proteins`) -> tmp_1
rbind(df,tmp_1) -> df_unfold
# 先把组别中有非小肽的注释蛋白的找出来
df_unfold %>% filter(grepl("^sp",Protein)) %>% filter(!Protein %in% uniprot.human.sep.id$V1) -> df_anno_pro
unique(df_anno_pro$Group_id) -> df_anno_pro_gi
# 特异性+非特异性总共有多少经典的SEP
df_unfold %>% filter(Protein %in% uniprot.human.sep.id$V1) -> df_unfold_target
n_distinct(df_unfold_target$Protein)
# 特异性+非特异性分别是哪些
df_unfold_target %>% group_by(Group_id) %>% filter(n()==1 & !Group_id %in% df_anno_pro_gi) -> df_unique
unique(df_unique$Group_id) -> df_unique_gi
df_unfold_target %>% group_by(Group_id) %>% filter(n()>1 | Group_id %in% df_anno_pro_gi) -> df_multiple
unique(df_multiple$Group_id) -> df_multiple_gi
# 合并结果
df_unfold_target %>% mutate(PSM_type=case_when(
    Group_id %in% df_unique_gi ~ "Unique",
    Group_id %in% df_multiple_gi ~ "NonUnique",
)) -> df_unfold_target_add_PSM_type
# 数量check
n_distinct(df_unfold_target_add_PSM_type$Protein)
df_unfold_target_add_PSM_type %>% group_by(PSM_type) %>% summarise(n_distinct=n_distinct(Protein))
# 导出
fwrite(df_unfold_target_add_PSM_type,o("psm_cano_sep_all.txt"),sep='\t')

[1] 2023

[1] 2023

PSM_type,n_distinct
<chr>,<int>
NonUnique,1479
Unique,838
